# Pan-tilt debug notebook

Talks directly to the positioner over serial using the PTCR-96 protocol (MN00162), bypassing `app.py` entirely (same pattern as `pantilt/pt_get_ping.py` in the original module).

**`app.py` must not be running at the same time** — a serial port can only be opened by one process. Stop the Flask app before running these cells, or stop the kernel before starting the app.

Run the first cell to connect, then run any of the other cells in any order/any number of times. Each cell prints the result so you can see immediately whether something failed.

Camera/lens/preset-table/tour/OSD commands aren't in `lib/qpt90.py` -- this app has no camera/lens hardware and doesn't use presets/tours. Everything else in the protocol is implemented: heater (Cell 10), firmware revision (Cell 11), comm timeout (Cell 12), max speed (Cell 13), ramp parameters (Cell 14), encoder align (Cell 15), homing cycle (Cell 16), identity (Cell 17). Cells 15/16 move the platform and block until they finish -- read their warnings before running them.

**Cells 18-20: mapping the pan/tilt keep-out zone.** The instrument's antennas can hit the horizontal mounting bar at some pan/tilt combinations (e.g. near tilt=90°) but not others (e.g. tilt=-90° is clear). Cell 18 is a fine, adjustable-step jog for creeping carefully toward the boundary while watching by eye; Cell 19 records the current position as a measured boundary sample and saves it to **`config/keepout/<INSTRUMENT_NAME>.json`** — one file per instrument/mount, since different antenna geometry means a different safe envelope. It reloads that file automatically, so samples accumulate across sessions, not just within one kernel run; changing `INSTRUMENT_NAME` mid-session switches to (or starts) a different file cleanly. Cell 20 undoes the last recorded sample if you misjudged one. Once mapped, pick the instrument from `app.py`'s Advanced Settings **"Keep-out profile"** dropdown to make it the active restriction — `controller.py` enforces both the target position and the path taken to reach it (see `lib/keepout.py`).

In [1]:
# Cell 1 -- imports + connect
# Leave PORT_HINT = "" to auto-scan every detected serial port, or set it
# to a specific port (e.g. "COM5") to connect directly.

from lib import qpt90

PORT_HINT = "COM5"
BAUD = 9600

print(f"Connecting (port={PORT_HINT or 'auto-scan'}, baud={BAUD})...")
port, dev = qpt90.find_qpt90(port_hint=PORT_HINT, baud=BAUD)

if dev is None:
    print("FAILED: no pan-tilt responded. Candidate ports tried:", qpt90.list_candidate_ports())
else:
    print(f"Connected on {port}")

Connecting (port=COM5, baud=9600)...
Connected on COM5


In [2]:
# Cell 2 -- get status

status = dev.get_status()
print(f"pan={status.pan_deg:.2f} deg  tilt={status.tilt_deg:.2f} deg")
print(f"moving={status.moving}  hard_limit={status.hard_limit}  soft_limit={status.soft_limit}  continuous={status.continuous}")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

pan=6.05 deg  tilt=4.00 deg
moving=False  hard_limit=False  soft_limit=False  continuous=False
faults: none


In [7]:
# Cell 3 -- turn pan +1 deg (relative move)
import time

DELTA_DEG = 1.0
MOVE_TIMEOUT_S = 15

before = dev.get_status()
print(f"pan before: {before.pan_deg:.1f} deg")

dev.move_delta(pan_deg=DELTA_DEG, tilt_deg=0.0)

deadline = time.time() + MOVE_TIMEOUT_S
status = dev.get_status()
while time.time() < deadline and status.moving:
    time.sleep(0.2)
    status = dev.get_status()

print(f"pan after:  {status.pan_deg:.1f} deg  (moving={status.moving})")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

pan before: 10.1 deg
pan after:  11.1 deg  (moving=False)
faults: none


In [18]:
# Cell 4 -- turn pan -1 deg (relative move)
import time

DELTA_DEG = -1.0
MOVE_TIMEOUT_S = 15

before = dev.get_status()
print(f"pan before: {before.pan_deg:.1f} deg")

dev.move_delta(pan_deg=DELTA_DEG, tilt_deg=0.0)

deadline = time.time() + MOVE_TIMEOUT_S
status = dev.get_status()
while time.time() < deadline and status.moving:
    time.sleep(0.2)
    status = dev.get_status()

print(f"pan after:  {status.pan_deg:.1f} deg  (moving={status.moving})")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

pan before: 1.0 deg
pan after:  -0.0 deg  (moving=False)
faults: none


In [ ]:
# Cell 5 -- turn tilt +1 deg (relative move)
import time

DELTA_DEG = 1.0
MOVE_TIMEOUT_S = 15

before = dev.get_status()
print(f"tilt before: {before.tilt_deg:.1f} deg")

dev.move_delta(pan_deg=0.0, tilt_deg=DELTA_DEG)

deadline = time.time() + MOVE_TIMEOUT_S
status = dev.get_status()
while time.time() < deadline and status.moving:
    time.sleep(0.2)
    status = dev.get_status()

print(f"tilt after:  {status.tilt_deg:.1f} deg  (moving={status.moving})")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

In [22]:
# Cell 6 -- turn tilt -1 deg (relative move)
import time

DELTA_DEG = -1.0
MOVE_TIMEOUT_S = 15

before = dev.get_status()
print(f"tilt before: {before.tilt_deg:.1f} deg")

dev.move_delta(pan_deg=0.0, tilt_deg=DELTA_DEG)

deadline = time.time() + MOVE_TIMEOUT_S
status = dev.get_status()
while time.time() < deadline and status.moving:
    time.sleep(0.2)
    status = dev.get_status()

print(f"tilt after:  {status.tilt_deg:.1f} deg  (moving={status.moving})")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

tilt before: 1.0 deg
tilt after:  0.0 deg  (moving=False)
faults: none


In [ ]:
# Cell 7 -- move to an absolute pan/tilt position
# Edit the two target angles below, then run this cell.
import time

TARGET_PAN_DEG = 0.0
TARGET_TILT_DEG = 0.0
MOVE_TIMEOUT_S = 30

print(f"Moving to pan={TARGET_PAN_DEG} deg, tilt={TARGET_TILT_DEG} deg ...")
dev.move_to(pan_deg=TARGET_PAN_DEG, tilt_deg=TARGET_TILT_DEG)

deadline = time.time() + MOVE_TIMEOUT_S
status = dev.get_status()
while time.time() < deadline and status.moving:
    time.sleep(0.2)
    status = dev.get_status()

print(f"now at: pan={status.pan_deg:.1f} deg  tilt={status.tilt_deg:.1f} deg  (moving={status.moving})")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

In [ ]:
# Cell 8 -- stop motion

status = dev.stop()
print(f"Stopped. pan={status.pan_deg:.1f} deg  tilt={status.tilt_deg:.1f} deg  moving={status.moving}")

In [ ]:
# Cell 9 -- clear latched faults (TO/DE)

status = dev.clear_faults()
print(f"faults after clear: {', '.join(status.faults) if status.faults else 'none'}")

In [ ]:
# Cell 10 -- get/set heater config (97H)
# HEATER_OFF=0 ("No Heat"), HEATER_SHARE=1 (off while moving), HEATER_FULL=2
# (concurrent with motion). Leave SET_TO = None to just query the current
# mode without changing it.
from lib import qpt90

SET_TO = None   # e.g. qpt90.HEATER_SHARE to change it

if SET_TO is not None:
    confirmed = dev.set_heater_config(SET_TO)
    print(f"Set heater to {SET_TO}, unit confirmed {confirmed}")
else:
    confirmed = dev.get_heater_config()
    print(f"Current heater config: {confirmed}")

names = {qpt90.HEATER_OFF: "No Heat", qpt90.HEATER_SHARE: "Share (off while moving)", qpt90.HEATER_FULL: "Full Heat"}
print(names.get(confirmed, "unknown"))

In [ ]:
# Cell 11 -- get firmware revision (9AH)

fw = dev.get_firmware_revision()
print(f"Firmware v{fw['major']}.{fw['minor']}, released {fw['year']:04d}-{fw['month']:02d}-{fw['day']:02d}")

In [ ]:
# Cell 12 -- get/set communication timeout (96H)
# Seconds before the positioner treats a lost link as a fault and stops
# itself (0 = disabled/"defeat"). app.py's controller.py always sets this to
# 2s on connect as a hardware safety backstop -- this cell is just to inspect
# or experiment with it directly. Leave SET_TO = None to just query.

SET_TO = None   # e.g. 2

if SET_TO is not None:
    confirmed = dev.set_comm_timeout(SET_TO)
    print(f"Set comm timeout to {SET_TO}s, unit confirmed {confirmed}s")
else:
    confirmed = dev.get_comm_timeout()
    print(f"Current comm timeout: {confirmed}s" + (" (disabled)" if confirmed == 0 else ""))

In [ ]:
# Cell 13 -- get/set maximum speed per axis (9CH)
# 1-255 per axis. app.py's controller.py already applies pan_max_speed/
# tilt_max_speed from Advanced Settings on every connect -- this cell is for
# inspecting or experimenting with it directly, including the STOR
# (non-volatile, survives power-cycle) option the dashboard doesn't expose.
# Leave SET_TO = None to just query the current (volatile) values.

SET_TO = None   # e.g. (100, 100) as (pan_max, tilt_max)
STORE = False   # True writes to non-volatile memory instead of just this session

if SET_TO is not None:
    pan_max, tilt_max = dev.set_max_speed(*SET_TO, stored=STORE)
    print(f"Set max speed to pan={SET_TO[0]} tilt={SET_TO[1]} (stored={STORE}), unit confirmed pan={pan_max} tilt={tilt_max}")
else:
    pan_max, tilt_max = dev.get_max_speed(stored=STORE)
    print(f"Current max speed ({'stored default' if STORE else 'volatile/session'}): pan={pan_max} tilt={tilt_max}")

In [ ]:
# Cell 14 -- get/set pan & tilt ramp parameters (92H)
# Accel/decel/start-stop speed tuning per axis (MN00162 Sec 2.9.4). These are
# platform/load-dependent and must be derived by testing -- don't guess.
# Leave SET_TO = None to just query the current values.

SET_TO = None   # e.g. dict(pan_start_stop=50, pan_acc_dec=40, pan_ramp=8,
                #           tilt_start_stop=50, tilt_acc_dec=40, tilt_ramp=8)

if SET_TO is not None:
    params = dev.set_ramp_params(**SET_TO)
else:
    params = dev.get_ramp_params()
print(params)

In [ ]:
# Cell 15 -- initial encoder align (9DH)
# CAUTION (MN00162 Sec 2.9.11): hard limits must be set correctly before
# running this. The platform will move and stop responding to ANY command
# until it finds the pan index pulse -- that's normal, not a hang. Jog to
# physical pan center by hand first (Cells 3/4), THEN run this cell.

result = dev.initial_encoder_align()
print(f"Encoder aligned. pan_index={result['pan_index_deg']:.2f} deg  tilt_index={result['tilt_index_deg']:.2f} deg")

In [ ]:
# Cell 16 -- perform homing cycle (9EH)
# CAUTION (MN00162 Sec 2.9.11/2.9.13): hard limits must be set correctly
# before running this. The platform will move at high speed hunting for the
# index pulse on BOTH axes and stop responding to ANY command until it
# returns to its original (corrected) position -- can take several seconds.

result = dev.perform_homing_cycle()
print(f"pan_index_found={result['pan_index_found']}  tilt_index_found={result['tilt_index_found']}")
print(f"pan_offset={result['pan_offset_deg']:.2f} deg  tilt_offset={result['tilt_offset_deg']:.2f} deg")

In [ ]:
# Cell 17 -- get/set RS-485 identity address (9FH)
# Only matters on a shared RS-485 daisy chain with multiple units -- this
# setup uses a dedicated link (identity 0), so there's normally no reason to
# touch this. Leave SET_TO = None to just query. If you do set a non-zero
# identity, this driver (dev.identity) updates itself to match automatically,
# but you'd need to pass identity=... to qpt90.find_qpt90() next time to
# reconnect to a unit that's no longer at the default identity 0.

SET_TO = None   # e.g. 5

if SET_TO is not None:
    confirmed = dev.set_identity(SET_TO)
    print(f"Set identity to {SET_TO}, unit confirmed {confirmed}")
else:
    confirmed = dev.get_identity()
    print(f"Current identity: {confirmed}")

In [ ]:
# Cell 18 -- fine jog with an adjustable step, for carefully exploring the
# pan/tilt keep-out zone near the horizontal mounting bar (the antennas can
# hit the bar at some combinations of pan and tilt). Re-run this cell
# repeatedly with a SMALL step, checking antenna clearance BY EYE after each
# move, before running it again. Never jump straight to a large delta near a
# suspected danger zone.

import time

AXIS = "pan"          # "pan" or "tilt"
STEP_DEG = 1.0         # start small (0.5-1.0 deg) once you suspect you're close
MOVE_TIMEOUT_S = 15

before = dev.get_status()
print(f"before: pan={before.pan_deg:.2f} deg  tilt={before.tilt_deg:.2f} deg")

if AXIS == "pan":
    dev.move_delta(pan_deg=STEP_DEG, tilt_deg=0.0)
else:
    dev.move_delta(pan_deg=0.0, tilt_deg=STEP_DEG)

deadline = time.time() + MOVE_TIMEOUT_S
status = dev.get_status()
while time.time() < deadline and status.moving:
    time.sleep(0.2)
    status = dev.get_status()

print(f"after:  pan={status.pan_deg:.2f} deg  tilt={status.tilt_deg:.2f} deg  (moving={status.moving})")
print(f"faults: {', '.join(status.faults) if status.faults else 'none'}")

In [ ]:
# Cell 19 -- record the current position as a measured keep-out boundary
# point, and persist it to config/keepout/<INSTRUMENT_NAME>.json right away
# (loaded automatically at the top of this cell next time, so samples
# accumulate across sessions, not just within one kernel run).
#
# Set INSTRUMENT_NAME once per instrument/mount -- each gets its own file,
# since different antenna geometry means a different safe envelope. Pick it
# from app.py's Advanced Settings "Keep-out profile" dropdown afterward to
# make it the active restriction.
#
# Workflow: pick a tilt angle, use Cell 18 to creep pan outward (positive
# direction) until the antennas are JUST about to touch the bar, then run
# this cell to log it. Repeat creeping pan in the NEGATIVE direction from
# center at the same tilt and log that boundary too -- don't assume the safe
# range is symmetric around pan=0. Repeat at a handful of different tilt
# angles (e.g. every 10-15 deg from a clearly-safe tilt like -90 up to a
# clearly-blocked one like 90) to map out the taper.
#
# Made a mistake? Cell 20 removes the last recorded sample.

import json
import os

INSTRUMENT_NAME = "default"   # e.g. "radar_v1", "lidar_unit" -- one file per instrument/mount

SAMPLES_PATH = os.path.join("config", "keepout", f"{INSTRUMENT_NAME}.json")

try:
    boundary_samples
    if _boundary_samples_instrument != INSTRUMENT_NAME:
        raise NameError   # INSTRUMENT_NAME changed since last run -- reload for the new one
except NameError:
    if os.path.exists(SAMPLES_PATH):
        with open(SAMPLES_PATH, "r") as f:
            boundary_samples = json.load(f)
        print(f"Loaded {len(boundary_samples)} existing sample(s) from {SAMPLES_PATH}")
    else:
        boundary_samples = []
    _boundary_samples_instrument = INSTRUMENT_NAME


def save_samples():
    os.makedirs(os.path.dirname(SAMPLES_PATH), exist_ok=True)
    with open(SAMPLES_PATH, "w") as f:
        json.dump(boundary_samples, f, indent=2)


status = dev.get_status()
boundary_samples.append({"tilt_deg": round(status.tilt_deg, 2), "pan_deg": round(status.pan_deg, 2)})
save_samples()

print(f"[{INSTRUMENT_NAME}] Recorded:", boundary_samples[-1])
print(f"Saved {len(boundary_samples)} total sample(s) to {SAMPLES_PATH}")
print("\nAll samples so far (sorted by tilt):")
for s in sorted(boundary_samples, key=lambda s: s["tilt_deg"]):
    print(f"  tilt={s['tilt_deg']:+.2f} deg  ->  boundary pan={s['pan_deg']:+.2f} deg")

print("\nJSON (paste into keepout_visualizer.html):")
print(json.dumps(boundary_samples))

In [ ]:
# Cell 20 -- undo the last recorded boundary sample for the CURRENT
# INSTRUMENT_NAME (misclick, wrong axis, whatever) and re-save. Run Cell 19
# at least once first in this session so boundary_samples/save_samples()/
# SAMPLES_PATH exist.

if boundary_samples:
    removed = boundary_samples.pop()
    save_samples()
    print(f"[{INSTRUMENT_NAME}] Removed:", removed)
    print(f"{len(boundary_samples)} sample(s) remain, saved to {SAMPLES_PATH}")
else:
    print("Nothing to remove.")

In [ ]:
# Cell 21 -- close the serial connection (run this last, when you're done)

dev.ser.close()
print("Serial connection closed.")